Install required Python libraries.

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece

Import necessary modules.

In [2]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    LoraConfig,
    get_peft_model
)

Load tokenizer and base model.

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

print("Model Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model Loaded Successfully!


Calculate and display parameters before LoRA.

In [4]:
total_before = sum(
    p.numel() for p in base_model.parameters()
)

trainable_before = sum(
    p.numel()
    for p in base_model.parameters()
    if p.requires_grad
)

percentage_before = (
    trainable_before / total_before
) * 100

print("===== BEFORE LoRA =====")
print(f"Total Parameters      : {total_before:,}")
print(f"Trainable Parameters  : {trainable_before:,}")
print(f"Trainable Percentage  : {percentage_before:.4f}%")

===== BEFORE LoRA =====
Total Parameters      : 1,100,048,384
Trainable Parameters  : 1,100,048,384
Trainable Percentage  : 100.0000%


Define LoRA configuration.

In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

print("LoRA Config Created!")

LoRA Config Created!


Apply LoRA to the base model.

In [6]:
lora_model = get_peft_model(
    base_model,
    lora_config
)

print("LoRA Applied Successfully!")

LoRA Applied Successfully!


Calculate and display parameters after LoRA.

In [7]:
total_after = sum(
    p.numel() for p in lora_model.parameters()
)

trainable_after = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

percentage_after = (
    trainable_after / total_after
) * 100

print("===== AFTER LoRA =====")
print(f"Total Parameters      : {total_after:,}")
print(f"Trainable Parameters  : {trainable_after:,}")
print(f"Trainable Percentage  : {percentage_after:.4f}%")

===== AFTER LoRA =====
Total Parameters      : 1,101,174,784
Trainable Parameters  : 1,126,400
Trainable Percentage  : 0.1023%


Compare parameters before and after LoRA.

In [8]:
print("\n===== PARAMETER COMPARISON =====\n")

print(
    f"{'Metric':<25}"
    f"{'Before LoRA':<20}"
    f"{'After LoRA':<20}"
)

print("-"*65)

print(
    f"{'Total Parameters':<25}"
    f"{total_before:<20,}"
    f"{total_after:<20,}"
)

print(
    f"{'Trainable Parameters':<25}"
    f"{trainable_before:<20,}"
    f"{trainable_after:<20,}"
)

print(
    f"{'Trainable Percentage':<25}"
    f"{percentage_before:<20.4f}"
    f"{percentage_after:<20.4f}"
)


===== PARAMETER COMPARISON =====

Metric                   Before LoRA         After LoRA          
-----------------------------------------------------------------
Total Parameters         1,100,048,384       1,101,174,784       
Trainable Parameters     1,100,048,384       1,126,400           
Trainable Percentage     100.0000            0.1023              


Print trainable parameters of the LoRA model.

In [9]:
lora_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Create a sample dataset.

In [10]:
from datasets import Dataset

data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 3
})


Display dataset examples.

In [11]:
for i in range(len(dataset)):
    print(f"\nExample {i+1}")
    print("Question:", dataset[i]["question"])
    print("Answer:", dataset[i]["answer"])


Example 1
Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.

Example 2
Question: How to treat a common cold?
Answer: Rest, hydration, and over-the-counter medications can help manage symptoms.

Example 3
Question: What are the symptoms of hypertension?
Answer: Symptoms include headaches, shortness of breath, and nosebleeds.


Format dataset examples for training.

In [12]:
def format_example(example):
    return {
        "text":
        f"Question: {example['question']}\n"
        f"Answer: {example['answer']}"
    }

dataset = dataset.map(format_example)

dataset[0]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'question': 'What is diabetes?',
 'answer': 'Diabetes is a chronic disease where the body cannot regulate blood sugar properly.',
 'text': 'Question: What is diabetes?\nAnswer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.'}

Tokenize the dataset.

In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Display input IDs for the first example.

In [14]:
print("Input IDs:\n")
print(tokenized_dataset[0]["input_ids"])

Input IDs:

[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


Display attention mask for the first example.

In [15]:
print("Attention Mask:\n")
print(tokenized_dataset[0]["attention_mask"])

Attention Mask:

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Show tokenized and original data for the first example.

In [16]:
print("Question:")
print(dataset[0]["question"])

print("\nAnswer:")
print(dataset[0]["answer"])

print("\nInput IDs:")
print(tokenized_dataset[0]["input_ids"])

print("\nAttention Mask:")
print(tokenized_dataset[0]["attention_mask"])

Question:
What is diabetes?

Answer:
Diabetes is a chronic disease where the body cannot regulate blood sugar properly.

Input IDs:
[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

Prepare tokenized dataset for training.

In [17]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["question", "answer", "text"]
)

tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"]}
)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Define training arguments.

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

Initialize the Hugging Face Trainer.

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset
)

Start model training.

In [20]:
trainer.train()

Step,Training Loss
1,8.105241
2,7.734685
3,6.114216
4,4.613855
5,3.354999
6,3.098489
7,3.147903
8,2.898071
9,3.059861


TrainOutput(global_step=9, training_loss=4.680813444985284, metrics={'train_runtime': 5.4916, 'train_samples_per_second': 1.639, 'train_steps_per_second': 1.639, 'total_flos': 7158335275008.0, 'train_loss': 4.680813444985284, 'epoch': 3.0})

Save the fine-tuned LoRA adapter.

In [21]:
lora_model.save_pretrained("medical_lora_adapter")

print("LoRA Adapter Saved Successfully!")

LoRA Adapter Saved Successfully!


Save the tokenizer.

In [22]:
tokenizer.save_pretrained("medical_lora_adapter")

('medical_lora_adapter/tokenizer_config.json',
 'medical_lora_adapter/chat_template.jinja',
 'medical_lora_adapter/tokenizer.json')

List contents of the saved adapter directory.

In [23]:
import os

print(os.listdir("medical_lora_adapter"))

['chat_template.jinja', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md']


Reload base model and load LoRA adapter for inference.

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

base_model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = PeftModel.from_pretrained(
    base_model,
    "medical_lora_adapter"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Generate response with base model (before LoRA).

In [25]:
question = "What is diabetes?"

prompt = f"Question: {question}\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7
)

response_before = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("===== BEFORE LoRA =====")
print(response_before)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BEFORE LoRA =====
Question: What is diabetes?
Answer: Diabetes is a chronic illness where the body cannot produce enough insulin or can't use the insulin it does produce properly. Insulin is a hormone that helps the body use sugar and other foods


Generate response with LoRA-tuned model (after LoRA).

In [26]:
question = "What is diabetes?"

prompt = f"Question: {question}\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7
)

response_after = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("===== AFTER LoRA =====")
print(response_after)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== AFTER LoRA =====
Question: What is diabetes?
Answer: Diabetes is a chronic condition in which the body's blood sugar levels are higher than normal due to a lack of insulin production or a lack of insulin function. This can lead to serious health problems, including blindness


Evaluate and compare responses.

In [27]:
print("\n========== EVALUATION ==========\n")

print("Question:")
print(question)

print("\nBefore LoRA (Generic Response):")
print(response_before)

print("\nAfter LoRA (Medical Domain Response):")
print(response_after)


========== EVALUATION ==========

Question:
What is diabetes?

Before LoRA (Generic Response):
Question: What is diabetes?
Answer: Diabetes is a chronic illness where the body cannot produce enough insulin or can't use the insulin it does produce properly. Insulin is a hormone that helps the body use sugar and other foods

After LoRA (Medical Domain Response):
Question: What is diabetes?
Answer: Diabetes is a chronic condition in which the body's blood sugar levels are higher than normal due to a lack of insulin production or a lack of insulin function. This can lead to serious health problems, including blindness
